# Chapter 44 — PyTorch Autograd

## Learning goals

Chapter 43 introduced CPU tensors and previewed automatic differentiation.

This chapter uses tiny calculations with known derivatives to explain PyTorch autograd in detail.

By the end of this chapter, you should be able to:

1. Create floating-point tensors that track gradients.
2. Build a forward computation and scalar loss.
3. Run `.backward()` and read leaf-tensor gradients from `.grad`.
4. Compare PyTorch gradients with derivatives computed by hand.
5. Explain and control gradient accumulation.
6. Update parameters without recording the update in the computation graph.
7. Use a fresh graph for each training step.
8. Connect PyTorch tensor autograd to the homemade `TrackedNumber` engine.

## The big idea

A floating-point tensor created with `requires_grad=True` asks PyTorch to record differentiable operations that depend on it.

Those operations form a computation graph during the forward pass.

Calling `.backward()` on a scalar output applies the chain rule from that output back to the tracked leaf tensors.

PyTorch stores each resulting leaf gradient in the tensor's `.grad` attribute.

## Terms used in this chapter

- **Autograd** is PyTorch's automatic differentiation system.
- A **computation graph** records how operations connect input tensors to later tensors.
- A **leaf tensor** is created directly rather than by a recorded operation.
- `requires_grad=True` requests gradient tracking for a floating-point or complex tensor.
- `.backward()` runs reverse-mode automatic differentiation.
- `.grad` stores an accumulated gradient on a tracked leaf tensor.
- **Gradient accumulation** means a new backward result is added to an existing `.grad`.
- `torch.no_grad()` temporarily disables operation recording.
- `.detach()` returns a tensor view of the same values without the current autograd history.

## Import PyTorch and stay on CPU

All tensors in this course remain on the CPU.

An explicit floating-point dtype keeps the examples independent of PyTorch's process-wide default dtype.

In [1]:
import torch

device = "cpu"

print("PyTorch version:", torch.__version__)
print("Course device:", device)

assert device == "cpu"

PyTorch version: 2.2.2
Course device: cpu


## Track one parameter

Start at the exact solution for the model `prediction = weight * input_number`.

Before backward, the leaf tensor has no stored gradient.

In [2]:
weight = torch.tensor(
    2.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
input_number = torch.tensor(5.0, dtype=torch.float32, device=device)
target_number = torch.tensor(10.0, dtype=torch.float32, device=device)

prediction = weight * input_number
loss = (prediction - target_number) ** 2

print("Weight:", weight)
print("Gradient before backward:", weight.grad)
print("Prediction:", prediction)
print("Loss:", loss)

loss.backward()

print("Gradient after backward:", weight.grad)

assert weight.is_leaf
assert weight.grad is not None
assert weight.grad.item() == 0.0

Weight: tensor(2., requires_grad=True)
Gradient before backward: None
Prediction: tensor(10., grad_fn=<MulBackward0>)
Loss: tensor(0., grad_fn=<PowBackward0>)
Gradient after backward: tensor(0.)


The loss is zero because the prediction already equals the target.

The derivative is also zero at this minimum.

## Move away from the minimum

Set the weight to `1.0` so the prediction is too low.

For squared error, the manual derivative with respect to the weight is:

```text
2 × (prediction - target) × input
```

In [3]:
weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
input_number = torch.tensor(5.0, dtype=torch.float32, device=device)
target_number = torch.tensor(10.0, dtype=torch.float32, device=device)

prediction = weight * input_number
loss = (prediction - target_number) ** 2
loss.backward()

manual_weight_gradient = (
    2.0 * (prediction.item() - target_number.item()) * input_number.item()
)

print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("PyTorch gradient:", weight.grad)
print("Manual gradient:", manual_weight_gradient)

assert weight.grad is not None
assert weight.grad.item() == -50.0
assert weight.grad.item() == manual_weight_gradient

Prediction: 5.0
Loss: 25.0
PyTorch gradient: tensor(-50.)
Manual gradient: -50.0


The negative gradient says that increasing the weight locally decreases the loss.

Calling `.item()` extracted Python numbers only for reporting and the independent manual check.

Using `.item()` inside a model computation would break that part of the computation graph.

## Check other known derivatives

A square and an expression with a shared input provide two compact checks.

The shared `second_number` contributes through multiplication and addition, so its two gradient contributions must be added.

In [4]:
number = torch.tensor(
    3.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
square = number**2
square.backward()

first_number = torch.tensor(
    2.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
second_number = torch.tensor(
    3.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
result = first_number * second_number + second_number
result.backward()  # type: ignore[no-untyped-call]

print("d(number²) / d(number):", number.grad)
print("Result:", result)
print("d(result) / d(first_number):", first_number.grad)
print("d(result) / d(second_number):", second_number.grad)

assert number.grad is not None
assert first_number.grad is not None
assert second_number.grad is not None
assert number.grad.item() == 6.0
assert first_number.grad.item() == 3.0
assert second_number.grad.item() == 3.0

d(number²) / d(number): tensor(6.)
Result: tensor(9., grad_fn=<AddBackward0>)
d(result) / d(first_number): tensor(3.)
d(result) / d(second_number): tensor(3.)


## Leaf tensors and graph-created tensors

Parameters such as `weight` are normally leaf tensors.

A prediction created from a parameter is not a leaf, although it still participates in the graph.

PyTorch populates `.grad` for tracked leaf tensors by default.

In [5]:
leaf_weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
graph_prediction = leaf_weight * input_number
graph_loss = (graph_prediction - target_number) ** 2

print("Weight is a leaf:", leaf_weight.is_leaf)
print("Prediction is a leaf:", graph_prediction.is_leaf)
print("Prediction requires gradients:", graph_prediction.requires_grad)
print("Prediction operation:", type(graph_prediction.grad_fn).__name__)
print("Loss operation:", type(graph_loss.grad_fn).__name__)

assert leaf_weight.is_leaf
assert not graph_prediction.is_leaf
assert graph_prediction.requires_grad
assert graph_prediction.grad_fn is not None

Weight is a leaf: True
Prediction is a leaf: False
Prediction requires gradients: True
Prediction operation: MulBackward0
Loss operation: PowBackward0


## Retain a non-leaf gradient when needed

Intermediate gradients are usually unnecessary for parameter training.

Calling `.retain_grad()` before backward asks PyTorch to preserve one for inspection.

In [6]:
retained_weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
retained_prediction = retained_weight * input_number
retained_prediction.retain_grad()
retained_loss = (retained_prediction - target_number) ** 2
retained_loss.backward()

print("Prediction gradient:", retained_prediction.grad)
print("Weight gradient:", retained_weight.grad)

assert retained_prediction.grad is not None
assert retained_weight.grad is not None
assert retained_prediction.grad.item() == -10.0
assert retained_weight.grad.item() == -50.0

Prediction gradient: tensor(-10.)
Weight gradient: tensor(-50.)


The loss derivative with respect to the prediction is `2 × (5 - 10) = -10`.

Multiplying by the input value `5` gives the weight gradient `-50`.

## Backward needs a scalar or an explicit starting gradient

Calling `.backward()` with no argument implicitly starts a scalar output with gradient `1`.

Training code therefore reduces per-example losses with an operation such as `.mean()` or `.sum()` before backward.

In [7]:
vector_weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
vector_inputs = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)
vector_targets = torch.tensor(
    [3.0, 5.0, 7.0],
    dtype=torch.float32,
    device=device,
)

per_example_losses = (vector_weight * vector_inputs - vector_targets) ** 2
scalar_average_loss = per_example_losses.mean()
scalar_average_loss.backward()

print("Per-example loss shape:", per_example_losses.shape)
print("Average-loss shape:", scalar_average_loss.shape)
print("Weight gradient:", vector_weight.grad)

assert per_example_losses.shape == torch.Size([3])
assert scalar_average_loss.shape == torch.Size([])
assert vector_weight.grad is not None

Per-example loss shape: torch.Size([3])
Average-loss shape: torch.Size([])
Weight gradient: tensor(-13.3333)


## Gradients accumulate

PyTorch adds each new backward result to any gradient already stored on a leaf.

Use a fresh forward graph for the second backward call while deliberately leaving the old gradient in place.

In [8]:
accumulating_weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)

first_loss = (accumulating_weight * 5.0 - 10.0) ** 2
first_loss.backward()
assert accumulating_weight.grad is not None
gradient_after_first_backward = accumulating_weight.grad.item()

second_loss = (accumulating_weight * 5.0 - 10.0) ** 2
second_loss.backward()
assert accumulating_weight.grad is not None
gradient_after_second_backward = accumulating_weight.grad.item()

print("After first backward:", gradient_after_first_backward)
print("After second backward:", gradient_after_second_backward)

assert gradient_after_first_backward == -50.0
assert gradient_after_second_backward == -100.0

After first backward: -50.0
After second backward: -100.0


Accumulation within one graph is essential when a value influences the loss along multiple paths.

Accumulation across ordinary training steps is usually unwanted, so training code clears old parameter gradients.

## Clear parameter gradients

Setting `.grad = None` is a simple and efficient way to indicate that no gradient is currently stored.

Later, an optimizer will perform this task with `optimizer.zero_grad()`.

In [9]:
def clear_gradients(parameters: list[torch.Tensor]) -> None:
    for parameter in parameters:
        parameter.grad = None


def gradient_of(parameter: torch.Tensor) -> torch.Tensor:
    if parameter.grad is None:
        raise RuntimeError("Call backward before requesting a gradient.")

    return parameter.grad


clear_gradients([accumulating_weight])

fresh_loss = (accumulating_weight * 5.0 - 10.0) ** 2
fresh_loss.backward()

print("Gradient after clearing and recomputing:", accumulating_weight.grad)

assert gradient_of(accumulating_weight).item() == -50.0

Gradient after clearing and recomputing: tensor(-50.)


Clearing `.grad` does not restore or preserve an old computation graph.

The fresh forward pass above created a new graph before the next backward pass.

## Differentiate with respect to two parameters

Add a bias to the familiar line model.

At `weight = 1`, `bias = 0`, `input = 5`, and `target = 10`, the expected gradients are `-50` for the weight and `-10` for the bias.

In [10]:
weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
bias = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)

prediction = weight * input_number + bias
loss = (prediction - target_number) ** 2

clear_gradients([weight, bias])
loss.backward()

manual_weight_gradient = (
    2.0 * (prediction.item() - target_number.item()) * input_number.item()
)
manual_bias_gradient = 2.0 * (prediction.item() - target_number.item())

print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("Weight gradient:", gradient_of(weight).item())
print("Bias gradient:", gradient_of(bias).item())

assert gradient_of(weight).item() == manual_weight_gradient == -50.0
assert gradient_of(bias).item() == manual_bias_gradient == -10.0

Prediction: 5.0
Loss: 25.0
Weight gradient: -50.0
Bias gradient: -10.0


## Update parameters without tracking the update

The parameter update is a training action rather than part of the model's forward function.

Place it inside `torch.no_grad()` so PyTorch does not add the update operations to the next computation graph.

In [11]:
learning_rate = 0.01
old_loss_value = loss.item()

with torch.no_grad():
    weight -= learning_rate * gradient_of(weight)
    bias -= learning_rate * gradient_of(bias)

new_prediction = weight * input_number + bias
new_loss = (new_prediction - target_number) ** 2

print("Updated weight:", weight.item())
print("Updated bias:", bias.item())
print("Old loss:", old_loss_value)
print("New loss:", new_loss.item())
print("Parameters remain leaves:", weight.is_leaf and bias.is_leaf)

assert new_loss.item() < old_loss_value
assert weight.is_leaf
assert bias.is_leaf
assert weight.requires_grad
assert bias.requires_grad

Updated weight: 1.5
Updated bias: 0.09999999403953552
Old loss: 25.0
New loss: 5.760000228881836
Parameters remain leaves: True


The gradients were negative, so subtracting them increased both parameters.

That raised the prediction toward the target and reduced the loss.

## Train the two-parameter model

A normal training step follows a fixed order:

1. Compute fresh predictions and loss.
2. Clear gradients left by the previous step.
3. Run backward.
4. Update parameters without gradient tracking.

This small loop applies that pattern to one example.

In [12]:
weight = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
bias = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
parameters = [weight, bias]
learning_rate = 0.01
number_of_steps = 10

print("step | weight | bias | prediction | loss")
print("-" * 52)

for step in range(number_of_steps):
    prediction = weight * input_number + bias
    loss = (prediction - target_number) ** 2

    clear_gradients(parameters)
    loss.backward()

    print(
        f"{step:>4} | "
        f"{weight.item():>7.4f} | "
        f"{bias.item():>7.4f} | "
        f"{prediction.item():>10.4f} | "
        f"{loss.item():>8.4f}"
    )

    with torch.no_grad():
        for parameter in parameters:
            parameter -= learning_rate * gradient_of(parameter)

assert loss.item() < 100.0

step | weight | bias | prediction | loss
----------------------------------------------------
   0 |  0.0000 |  0.0000 |     0.0000 | 100.0000
   1 |  1.0000 |  0.2000 |     5.2000 |  23.0400
   2 |  1.4800 |  0.2960 |     7.6960 |   5.3084
   3 |  1.7104 |  0.3421 |     8.8941 |   1.2231
   4 |  1.8210 |  0.3642 |     9.4692 |   0.2818
   5 |  1.8741 |  0.3748 |     9.7452 |   0.0649
   6 |  1.8996 |  0.3799 |     9.8777 |   0.0150
   7 |  1.9118 |  0.3824 |     9.9413 |   0.0034
   8 |  1.9177 |  0.3835 |     9.9718 |   0.0008
   9 |  1.9205 |  0.3841 |     9.9865 |   0.0002


The loss decreases because each step uses gradients from its own fresh forward pass.

Many weight-and-bias pairs fit this single example, so this example does not identify one unique pair.

## Vectorize a tiny dataset

Use three examples whose target rule is `target = 2 × input + 1`.

PyTorch can calculate all predictions at once and reduce their squared errors to one scalar mean loss.

In [13]:
input_numbers = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float32,
    device=device,
)
target_numbers = torch.tensor(
    [3.0, 5.0, 7.0],
    dtype=torch.float32,
    device=device,
)
weight = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
bias = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)

predictions = weight * input_numbers + bias
errors = predictions - target_numbers
average_loss = (errors**2).mean()

clear_gradients([weight, bias])
average_loss.backward()

manual_errors = predictions.detach() - target_numbers
dataset_manual_weight_gradient = (2.0 * manual_errors * input_numbers).mean()
dataset_manual_bias_gradient = (2.0 * manual_errors).mean()

print("Predictions:", predictions)
print("Average loss:", average_loss)
print("PyTorch weight gradient:", weight.grad)
print("Manual weight gradient:", dataset_manual_weight_gradient)
print("PyTorch bias gradient:", bias.grad)
print("Manual bias gradient:", dataset_manual_bias_gradient)

assert average_loss.shape == torch.Size([])
assert weight.grad is not None
assert bias.grad is not None
torch.testing.assert_close(weight.grad, dataset_manual_weight_gradient)
torch.testing.assert_close(bias.grad, dataset_manual_bias_gradient)

Predictions: tensor([0., 0., 0.], grad_fn=<AddBackward0>)
Average loss: tensor(27.6667, grad_fn=<MeanBackward0>)
PyTorch weight gradient: tensor(-22.6667)
Manual weight gradient: tensor(-22.6667)
PyTorch bias gradient: tensor(-10.)
Manual bias gradient: tensor(-10.)


The `.mean()` operation participates in the graph, so its division by the example count is included in both gradients.

The detached prediction values are used only for an independent manual calculation.

## Train on the tiny dataset

Run 100 fresh forward and backward passes.

Store Python numbers in the history so old graphs are not retained accidentally.

In [14]:
weight = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
bias = torch.tensor(
    0.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
parameters = [weight, bias]
learning_rate = 0.05
number_of_steps = 100
training_history: list[dict[str, float]] = []

for step in range(number_of_steps):
    predictions = weight * input_numbers + bias
    average_loss = ((predictions - target_numbers) ** 2).mean()

    clear_gradients(parameters)
    average_loss.backward()

    training_history.append(
        {
            "step": float(step),
            "weight": weight.item(),
            "bias": bias.item(),
            "loss": average_loss.item(),
            "weight_gradient": gradient_of(weight).item(),
            "bias_gradient": gradient_of(bias).item(),
        }
    )

    with torch.no_grad():
        for parameter in parameters:
            parameter -= learning_rate * gradient_of(parameter)

final_predictions = weight * input_numbers + bias
final_loss = ((final_predictions - target_numbers) ** 2).mean()

print("Final weight:", weight.item())
print("Final bias:", bias.item())
print("Final predictions:", final_predictions)
print("Final loss:", final_loss.item())

assert training_history[-1]["loss"] < training_history[0]["loss"]
assert final_loss.item() < 0.01
assert abs(weight.item() - 2.0) < 0.1
assert abs(bias.item() - 1.0) < 0.2

Final weight: 2.0132205486297607
Final bias: 0.9699465036392212
Final predictions: tensor([2.9832, 4.9964, 7.0096], grad_fn=<AddBackward0>)
Final loss: 0.0001295680267503485


The learned values approach the expected weight `2` and bias `1`.

The dataset contains enough distinct inputs to constrain both parameters.

## Inspect early training records

The first few records show how the scalar loss and both gradients change together.

In [15]:
print("step | weight | bias | loss | weight gradient | bias gradient")
print("-" * 75)

for row in training_history[:5]:
    print(
        f"{int(row['step']):>4} | "
        f"{row['weight']:>7.4f} | "
        f"{row['bias']:>7.4f} | "
        f"{row['loss']:>8.4f} | "
        f"{row['weight_gradient']:>15.4f} | "
        f"{row['bias_gradient']:>13.4f}"
    )

step | weight | bias | loss | weight gradient | bias gradient
---------------------------------------------------------------------------
   0 |  0.0000 |  0.0000 |  27.6667 |        -22.6667 |      -10.0000
   1 |  1.1333 |  0.5000 |   5.4885 |        -10.0889 |       -4.4667
   2 |  1.6378 |  0.7233 |   1.0897 |         -4.4874 |       -2.0022
   3 |  1.8621 |  0.8234 |   0.2172 |         -1.9928 |       -0.9045
   4 |  1.9618 |  0.8687 |   0.0441 |         -0.8819 |       -0.4155


## A graph normally supports one backward pass

Backward frees saved intermediate values that are no longer needed.

Calling backward again on the same graph normally fails, while recomputing the forward pass creates a usable new graph.

In [16]:
graph_weight = torch.tensor(
    1.0,
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
old_graph_loss = (graph_weight * 5.0 - 10.0) ** 2
old_graph_loss.backward()

try:
    old_graph_loss.backward()
except RuntimeError:
    print("A second backward on the same freed graph failed as expected.")
else:
    raise AssertionError("The reused graph should not support another backward.")

graph_weight.grad = None
fresh_graph_loss = (graph_weight * 5.0 - 10.0) ** 2
fresh_graph_loss.backward()

print("Fresh-graph gradient:", graph_weight.grad)

assert graph_weight.grad is not None
assert graph_weight.grad.item() == -50.0

A second backward on the same freed graph failed as expected.
Fresh-graph gradient: tensor(-50.)


Advanced code can request graph retention, but ordinary training should recompute the forward pass instead.

A loss belongs to the parameter values and operations from one particular forward pass.

## Integer token IDs select differentiable embeddings

Integer token IDs are lookup keys and cannot receive ordinary autograd gradients.

The floating-point embedding table selected by those IDs can receive gradients.

In [17]:
embedding_table = torch.tensor(
    [
        [0.10, 0.20],
        [0.30, 0.40],
        [0.50, 0.60],
    ],
    dtype=torch.float32,
    device=device,
    requires_grad=True,
)
token_ids = torch.tensor(
    [0, 2],
    dtype=torch.long,
    device=device,
)

selected_embeddings = embedding_table[token_ids]
embedding_loss = selected_embeddings.sum()
embedding_loss.backward()

expected_embedding_gradient = torch.tensor(
    [
        [1.0, 1.0],
        [0.0, 0.0],
        [1.0, 1.0],
    ],
    dtype=torch.float32,
    device=device,
)

print("Token IDs:", token_ids)
print("Selected embeddings:")
print(selected_embeddings)
print("Embedding-table gradient:")
print(embedding_table.grad)
print("Token-ID gradient:", token_ids.grad)

assert embedding_table.grad is not None
assert torch.equal(embedding_table.grad, expected_embedding_gradient)
assert token_ids.grad is None
assert not token_ids.requires_grad

Token IDs: tensor([0, 2])
Selected embeddings:
tensor([[0.1000, 0.2000],
        [0.5000, 0.6000]], grad_fn=<IndexBackward0>)
Embedding-table gradient:
tensor([[1., 1.],
        [0., 0.],
        [1., 1.]])
Token-ID gradient: None


Only rows `0` and `2` were selected, so only those embedding rows received nonzero gradients.

The token IDs identify the rows but are not themselves learnable numerical quantities.

## Connect PyTorch to homemade autograd

The two systems share the same conceptual parts:

| Homemade `TrackedNumber` | PyTorch tensor autograd |
|---|---|
| Scalar `.data` | Tensor values |
| Scalar `.gradient` | Leaf tensor `.grad` |
| Stored previous values | Computation graph |
| Local backward function | Operation-specific derivative rule |
| `result.backward()` | `loss.backward()` |

PyTorch extends the idea to multidimensional tensors, many operations, efficient kernels, and production-scale models.

## What not to do

- Do not forget `requires_grad=True` on floating-point parameters that should receive gradients.
- Do not expect `.grad` before backward.
- Do not leave old gradients in place between ordinary training steps.
- Do not update parameters while autograd is recording.
- Do not reuse an old loss after parameter values change.
- Do not call `.item()` or `.detach()` inside a computation that must remain differentiable.
- Do not try to differentiate integer token IDs.
- Do not add CUDA assumptions to this CPU-only course.

## Gotchas

- Gradient accumulation and graph retention are different concepts.
- Clearing `.grad` removes stored leaf gradients but does not rebuild a freed graph.
- A no-argument `.backward()` call expects a scalar output.
- Non-leaf gradients are not retained by default.
- Parameter updates belong inside `torch.no_grad()`.
- Every ordinary training step needs a fresh forward graph.

## Takeaways

PyTorch records tensor operations that depend on tracked floating-point parameters.

A scalar loss starts the backward pass, and leaf parameter gradients appear in `.grad`.

Those gradients match the derivatives computed by hand in the tiny examples.

The reusable training pattern is:

1. Compute a fresh forward pass and scalar loss.
2. Clear old parameter gradients.
3. Call `loss.backward()`.
4. Update parameters inside `torch.no_grad()`.
5. Repeat.

PyTorch autograd is the tensor-scale version of the chain-rule bookkeeping implemented by `TrackedNumber`.

## What comes next

The next chapter will organize trainable tensors inside PyTorch modules.

Modules remove the need to manage separate `weight` and `bias` tensors by hand and prepare the course for standard neural-network code.